In [27]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

In [28]:
df = pd.read_csv("../../output/processed_data/03_aspi_handled.csv")

# Convert to correct types
df['date'] = pd.to_datetime(df['date'])

# Important numeric columns
df['close'] = pd.to_numeric(df['close'], errors='coerce')
df['aspi_close'] = pd.to_numeric(df['aspi_close'], errors='coerce')

# Sort properly
df = df.sort_values(['symbol', 'date'])

In [29]:
df.groupby('symbol')[['symbol', 'date']].head()

,symbol,date
37119,AAIC.N0000,2025-02-03
37120,AAIC.N0000,2025-02-05
37121,AAIC.N0000,2025-02-06
37122,AAIC.N0000,2025-02-07
37123,AAIC.N0000,2025-02-10
...,...,...
36862,WAPO.N0000,2025-02-03
36863,WAPO.N0000,2025-02-05
36864,WAPO.N0000,2025-02-06
36865,WAPO.N0000,2025-02-07


In [30]:
# Stock return
df['R_i'] = df.groupby('symbol')['close'].pct_change()

# Market return (same for all stocks per day)
##df['R_m'] = df.groupby('symbol')['aspi_close'].pct_change() --bug


#bug fix
# keep one row per trading date cause not every stock traded everyday.
# so we're taking aspi for unique day


aspi_returns = (
    df[['date', 'aspi_close']]
    .drop_duplicates(subset='date')   # keep one row per trading date
    .sort_values('date')
    .assign(R_m=lambda x: x['aspi_close'].pct_change())
    [['date', 'R_m']]
)

if 'R_m' in df.columns:
    df = df.drop(columns=['R_m'])

df = df.merge(aspi_returns, on='date', how='left')


# Sanity check: R_m must be identical for all stocks on any given date
rm_per_date = df.groupby('date')['R_m'].nunique()
assert (rm_per_date <= 1).all(), "R_m is not unique per date — merge failed"
print(f"✓ R_m correctly computed across {rm_per_date.index.nunique()} trading days")
print(f"  NaN R_m rows (first trading day, expected): {df['R_m'].isna().sum()}")


✓ R_m correctly computed across 257 trading days
  NaN R_m rows (first trading day, expected): 202


In [31]:
df[['date', 'aspi_close']]

,date,aspi_close
0,2025-02-03,16956.49
1,2025-02-05,16456.10
2,2025-02-06,16506.73
3,2025-02-07,16734.68
4,2025-02-10,16566.27
...,...,...
51984,2026-02-23,23783.02
51985,2026-02-24,23690.62
51986,2026-02-25,23703.10
51987,2026-02-26,23781.23


In [32]:
df.head()

,symbol,date,open,high,low,close,volume,base_symbol,sector,suffix_cat,aspi_open,aspi_high,aspi_low,aspi_close,R_i,R_m
0,AAIC.N0000,2025-02-03,81.0,82.3,78.0,79.5,358254.0,AAIC,Insurance,0,17122.73,17127.65,16900.36,16956.49,NaN,NaN
1,AAIC.N0000,2025-02-05,79.0,79.6,73.1,76.0,646093.0,AAIC,Insurance,0,16956.49,16974.58,16440.74,16456.10,-0.044025,-0.029510
2,AAIC.N0000,2025-02-06,76.0,79.4,75.0,78.8,721181.0,AAIC,Insurance,0,16456.10,16675.80,16295.31,16506.73,0.036842,0.003077
3,AAIC.N0000,2025-02-07,79.1,80.5,78.6,79.0,289638.0,AAIC,Insurance,0,16506.73,16746.78,16506.38,16734.68,0.002538,0.013810
4,AAIC.N0000,2025-02-10,79.0,79.4,76.4,77.0,220182.0,AAIC,Insurance,0,16734.68,16794.64,16537.98,16566.27,-0.025316,-0.010064


In [33]:
event_date = pd.to_datetime("2025-11-28")


#bug -- goes from calender days ----------------------------------
#df['day'] = (df['date'] - event_date).dt.days 
# Estimation window
#estimation_df = df[(df['day'] >= -120) & (df['day'] <= -6)]


#fix-------------------------------------

trading_dates = sorted(df['date'].unique())           # all real trading days
event_idx     = trading_dates.index(event_date)       # position of Day 0

# Map each trading date to its trading-day offset from Day 0
date_to_td = {d: i - event_idx for i, d in enumerate(trading_dates)}
df['day']   = df['date'].map(date_to_td)

# Sanity check
print(f"Day  0  = {trading_dates[event_idx].date()}   (should be 2025-11-28)")
print(f"Day -5  = {trading_dates[event_idx - 5].date()}  (first day of event window)")
print(f"Day -6  = {trading_dates[event_idx - 6].date()}  (last day of estimation window)")
print(f"Day +30 = {trading_dates[event_idx + 30].date()} (last day of event window)")

# Estimation window
estimation_df = df[(df['day'] >= -120) & (df['day'] <= -6)]

Day  0  = 2025-11-28   (should be 2025-11-28)
Day -5  = 2025-11-21  (first day of event window)
Day -6  = 2025-11-20  (last day of estimation window)
Day +30 = 2026-01-14 (last day of event window)


In [34]:
#Entire research quality depends on these alpha,beta values

#There are three standard checks every OLS regression should report
# R² (R-squared)
"""
Measures how well the market model actually explains each stock's returns.
A value of 1.0 = perfect fit, 0.0 = the market explains nothing about this stock's movement. '
'Running the diagnostics on your actual data reveals the mean R² across all 286 stocks is only 0.08 — meaning the ASPI index explains just 8% of the typical stock's daily movement on average.
This is a genuinely important finding for your paper — it tells the reader that CSE stocks are largely idiosyncratic (driven by company-specific news rather than market-wide movements), which is characteristic of thin frontier markets. 
Without reporting R², this finding is invisible."""

# Durbin-Watson statistic
"""Durbin-Watson statistic
Tests whether the residuals (the leftover errors after fitting) are correlated with each other across time. 
The ideal value is 2.0, meaning no autocorrelation. 
Values below 1.5 suggest positive autocorrelation (today's error predicts tomorrow's error), 
which would mean the regression is missing some systematic pattern. 
From your data: 10 stocks have DW < 1.5 — their residuals are autocorrelated and their alpha/beta estimates are less reliable."""

# Jarque-Bera test
"""Tests whether the residuals follow a normal distribution, which OLS formally assumes. 
From your data: 264 out of 286 stocks (92%) have non-normal residuals. 
This is expected for stock returns — especially in emerging markets with fat tails — and is exactly why your paper also uses the Wilcoxon signed-rank test as a backup. 
But you need to report this number to justify that choice."""





"""results = []

for symbol, group in estimation_df.groupby('symbol'):

    # Drop NaNs
    group = group[['R_i', 'R_m']].dropna()

    if len(group) < 30:  # safety check
        continue

    X = group['R_m']
    y = group['R_i']

    # Add constant (for alpha)
    X = sm.add_constant(X)

    model = sm.OLS(y, X).fit()

    alpha = model.params['const']
    beta = model.params['R_m']

    results.append({
        'symbol': symbol,
        'alpha': alpha,
        'beta': beta
    })

alpha_beta_df = pd.DataFrame(results)"""

"results = []\n\nfor symbol, group in estimation_df.groupby('symbol'):\n\n    # Drop NaNs\n    group = group[['R_i', 'R_m']].dropna()\n\n    if len(group) < 30:  # safety check\n        continue\n\n    X = group['R_m']\n    y = group['R_i']\n\n    # Add constant (for alpha)\n    X = sm.add_constant(X)\n\n    model = sm.OLS(y, X).fit()\n\n    alpha = model.params['const']\n    beta = model.params['R_m']\n\n    results.append({\n        'symbol': symbol,\n        'alpha': alpha,\n        'beta': beta\n    })\n\nalpha_beta_df = pd.DataFrame(results)"

In [35]:
results = []

for symbol, group in estimation_df.groupby('symbol'):

    group = group[['R_i', 'R_m']].dropna()

    if len(group) < 30:
        continue

    X = sm.add_constant(group['R_m'])
    y = group['R_i']

    model = sm.OLS(y, X).fit()

    # ── OLS diagnostics ───────────────────────────────────────────────────────
    # R²: how much of the stock's return variance the market model explains
    r_squared = model.rsquared

    # Durbin-Watson: tests for autocorrelation in residuals
    # Ideal = 2.0  |  < 1.5 = positive autocorr  |  > 2.5 = negative autocorr
    dw_stat = sm.stats.stattools.durbin_watson(model.resid)

    # Jarque-Bera: tests whether residuals are normally distributed
    # p < 0.05 means residuals are non-normal (common in stock return data)
    jb_stat, jb_pvalue, _, _ = sm.stats.stattools.jarque_bera(model.resid)

    results.append({
        'symbol'    : symbol,
        'alpha'     : model.params['const'],
        'beta'      : model.params['R_m'],
        'r_squared' : r_squared,
        'dw_stat'   : dw_stat,
        'jb_pvalue' : jb_pvalue,
        'n_obs'     : len(group),       # how many trading days went into this regression
    })

alpha_beta_df = pd.DataFrame(results)

# ── Diagnostic summary ────────────────────────────────────────────────────────
print(f"Stocks with qualifying regressions : {len(alpha_beta_df)}")
print()
print(f"R² distribution:")
print(f"  Mean   : {alpha_beta_df['r_squared'].mean():.4f}")
print(f"  Median : {alpha_beta_df['r_squared'].median():.4f}")
print(f"  Max    : {alpha_beta_df['r_squared'].max():.4f}")
print()
print(f"Durbin-Watson:")
print(f"  Mean   : {alpha_beta_df['dw_stat'].mean():.3f}  (ideal = 2.0)")
print(f"  Stocks with DW < 1.5 (autocorrelation concern): "
      f"{(alpha_beta_df['dw_stat'] < 1.5).sum()}")
print()
print(f"Jarque-Bera normality test:")
print(f"  Stocks with non-normal residuals (p < 0.05): "
      f"{(alpha_beta_df['jb_pvalue'] < 0.05).sum()} / {len(alpha_beta_df)}"
      f"  → justifies using Wilcoxon as backup test")

Stocks with qualifying regressions : 204

R² distribution:
  Mean   : 0.1014
  Median : 0.0691
  Max    : 0.5090

Durbin-Watson:
  Mean   : 2.053  (ideal = 2.0)
  Stocks with DW < 1.5 (autocorrelation concern): 10

Jarque-Bera normality test:
  Stocks with non-normal residuals (p < 0.05): 195 / 204  → justifies using Wilcoxon as backup test


In [36]:
alpha_beta_df.head()

,symbol,alpha,beta,r_squared,dw_stat,jb_pvalue,n_obs
0,AAIC.N0000,-0.001283,1.025115,0.120007,1.857244,6.833979e-19,115
1,ABL.N0000,0.001316,0.572247,0.049462,1.807949,5.983976e-70,115
2,ACAP.N0000,0.002133,1.008940,0.020589,1.371832,9.692727e-128,112
3,ACL.N0000,0.001333,0.903848,0.144392,1.860860,1.828648e-03,115
4,AEL.N0000,0.000780,1.286056,0.204829,1.701587,1.163166e-18,115


In [37]:
df = df.merge(alpha_beta_df, on='symbol', how='left')
df.head()

,symbol,date,open,high,low,close,volume,base_symbol,sector,suffix_cat,...,aspi_close,R_i,R_m,day,alpha,beta,r_squared,dw_stat,jb_pvalue,n_obs
0,AAIC.N0000,2025-02-03,81.0,82.3,78.0,79.5,358254.0,AAIC,Insurance,0,...,16956.49,NaN,NaN,-196,-0.001283,1.025115,0.120007,1.857244,6.833979e-19,115.0
1,AAIC.N0000,2025-02-05,79.0,79.6,73.1,76.0,646093.0,AAIC,Insurance,0,...,16456.10,-0.044025,-0.029510,-195,-0.001283,1.025115,0.120007,1.857244,6.833979e-19,115.0
2,AAIC.N0000,2025-02-06,76.0,79.4,75.0,78.8,721181.0,AAIC,Insurance,0,...,16506.73,0.036842,0.003077,-194,-0.001283,1.025115,0.120007,1.857244,6.833979e-19,115.0
3,AAIC.N0000,2025-02-07,79.1,80.5,78.6,79.0,289638.0,AAIC,Insurance,0,...,16734.68,0.002538,0.013810,-193,-0.001283,1.025115,0.120007,1.857244,6.833979e-19,115.0
4,AAIC.N0000,2025-02-10,79.0,79.4,76.4,77.0,220182.0,AAIC,Insurance,0,...,16566.27,-0.025316,-0.010064,-192,-0.001283,1.025115,0.120007,1.857244,6.833979e-19,115.0


In [38]:
df.isnull().sum()

symbol           0
date             0
open             0
high             0
low              0
close            0
volume           0
base_symbol      0
sector           0
suffix_cat       0
aspi_open        0
aspi_high        0
aspi_low         0
aspi_close       0
R_i            206
R_m            202
day              0
alpha          149
beta           149
r_squared      149
dw_stat        149
jb_pvalue      149
n_obs          149
dtype: int64

In [39]:
df.to_csv('../../output/processed_data/04_Regression_handled.csv',index = False)